# 03 — Profiling Basics

> Companion to **Week 11**, Part 3 of the lecture.

## The story

You wrote a tiny ML web service. Every request takes 2.5 seconds. That feels
slow. **Where** is the time going? Don't guess — *measure*.

In this notebook we will:

1. Train a small classifier and save it to disk.
2. Time things three different ways: `time.time()`, `%timeit`, and `cProfile`.
3. Use `cProfile` to find the *exact* slow line — not "the code", a specific line.
4. Fix the bug and measure the speedup.


## Step 1 — Train and save a small model

This is the kind of artifact a real web service would load on startup.


In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

artifacts = Path("/tmp/week11_profiling")
artifacts.mkdir(exist_ok=True)

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data, digits.target, test_size=0.2, random_state=42
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

joblib.dump(model, artifacts / "model.pkl")
pd.DataFrame(X_train).to_csv(artifacts / "data.csv", index=False)

sample = X_test[0]
print("Saved model.pkl and data.csv to", artifacts)


## Step 2 — A slow function and a fast function

The **slow** function reloads the CSV and the model on every call. This is a
real bug in many FastAPI / Flask apps — people accidentally put `joblib.load(...)`
*inside* the request handler.

The **fast** function loads the model **once** at module level and reuses it.


In [ ]:
def slow_predict(sample):
    data = pd.read_csv(artifacts / "data.csv")     # reloaded every call!
    model = joblib.load(artifacts / "model.pkl")    # reloaded every call!
    return model.predict([sample])[0]


# Cache the model ONCE, outside the function
cached_model = joblib.load(artifacts / "model.pkl")


def fast_predict(sample):
    return cached_model.predict([sample])[0]


## Step 3 — Three ways to time code

### (a) `time.time()` — the stopwatch

Quick and dirty. Works anywhere. One single run, so the number is noisy.


In [ ]:
import time

start = time.time()
slow_predict(sample)
print(f"slow_predict took {time.time() - start:.3f} s (one run, noisy)")


### (b) IPython `%timeit` magic — the right way in Jupyter

`%timeit` runs the line many times, throws out outliers, and reports a stable
mean ± std. It even picks a sensible number of repetitions for you.

- `%timeit  expr`         → time a single expression
- `%%timeit`              → time the whole cell
- `%timeit -n 100 expr`   → force 100 runs per loop


In [ ]:
%timeit slow_predict(sample)


In [ ]:
%timeit fast_predict(sample)


You should see something like:

```
slow_predict:  ~30 ms ± 2 ms per loop
fast_predict:  ~50 µs ± 5 µs per loop      ← microseconds, not milliseconds
```

That's already a ~600× difference. But `%timeit` only tells us **how long** —
not **why**. For that we need `cProfile`.


### (c) `cProfile` — the itemized receipt

`cProfile` records every function call and how much time it took. The raw
output is huge and noisy, so we route it through `pstats` to:

1. Sort by `cumulative` time (slowest things first).
2. Print only the **top 10** lines.
3. Strip the long Python paths so the function names are readable.


In [ ]:
import cProfile
import pstats

profiler = cProfile.Profile()
profiler.enable()
for _ in range(20):                  # 20 runs so the numbers are stable
    slow_predict(sample)
profiler.disable()

stats = pstats.Stats(profiler)
stats.strip_dirs().sort_stats("cumulative").print_stats(10)


### How to read the output

Each row is one function. The columns you actually care about:

| Column | Meaning |
|---|---|
| `ncalls` | how many times the function was called |
| `tottime` | time spent **inside** this function (not its children) |
| `cumtime` | time spent inside this function **plus everything it called** |
| `filename:lineno(function)` | which line of which file |

**What to look for:**

- `slow_predict` itself has a huge `cumtime` — it's the entry point.
- Inside it, the biggest `cumtime` rows belong to **`read_csv`** and
  **`joblib.load`** (or `pickle.load`).
- `predict` is barely visible. **It is not the bottleneck.**

> 🩺 **Diagnosis:** ~99% of the time is spent re-loading the CSV and the
> pickled model on every call. The actual ML inference is microseconds.


## Step 4 — Measure the speedup we get from the fix

The fix ("load once at startup") is just two lines moved out of the function.
Let's measure how big that speedup is.


In [ ]:
import timeit

slow_time = timeit.timeit(lambda: slow_predict(sample), number=10) / 10
fast_time = timeit.timeit(lambda: fast_predict(sample), number=1000) / 1000

print(f"slow_predict: {slow_time * 1000:>10.3f} ms per call")
print(f"fast_predict: {fast_time * 1000:>10.3f} ms per call")
print(f"speedup     : {slow_time / fast_time:>10.1f} ×")


### Reflection

You should see a **huge** speedup — typically 100× or more. We did not change
the model. We did not change the prediction logic. We just **moved two lines
out of the request handler**.

> 🧠 **Take-away:** The first place to look for an ML web speedup is almost
> never the model itself — it's loading and copying work you didn't need to redo.

## 🧪 Your turn

1. Write a `medium_predict` that caches the model but still calls
   `pd.read_csv(...)` on every request. Where does it land between slow and fast?
2. Re-profile `medium_predict` with `cProfile`. Does the top of the report change?
3. **Bonus:** time `model.predict([sample])[0]` directly with `%timeit`. How
   much of `fast_predict` is "real" inference vs Python overhead?
